# Latihan Mandiri 8 - Primitif Sinkronisasi

**Nama:** Hanifah Salsabila Zuheri

**NIM:** 2411513016 

**Mata kuliah:** Pemrograman Paralel

## 1. RLock pada Class 

Buat sebuah class RekeningBank dengan method setor(jumlah) dan tarik(jumlah) , serta method
transfer(rekening_tujuan, jumlah) yang memanggil tarik() pada dirinya sendiri dan setor() pada rekening tujuan. Gunakan
RLock pada setiap instance. Jelaskan mengapa RLock diperlukan di sini dibanding Lock biasa (hint: pikirkan method apa yang saling memanggil
method lain pada objek yang sama).

In [1]:
import threading

class RekeningBank:
    def __init__(self, nama, saldo_awal=0):
        self.nama = nama
        self.saldo = saldo_awal
        self.lock = threading.RLock()

    def setor(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah setoran harus lebih dari 0")
        with self.lock:
            self.saldo += jumlah
            return self.saldo

    def tarik(self, jumlah):
        if jumlah <= 0:
            raise ValueError("Jumlah penarikan harus lebih dari 0")
        with self.lock:
            if self.saldo < jumlah:
                raise ValueError(f"Saldo rekening {self.nama} tidak mencukupi")
            self.saldo -= jumlah
            return self.saldo

    def transfer(self, rekening_tujuan, jumlah):
        if not isinstance(rekening_tujuan, RekeningBank):
            raise TypeError("Tujuan harus berupa objek RekeningBank")
        if rekening_tujuan is self:
            raise ValueError("Tidak bisa transfer ke rekening sendiri")
        if jumlah <= 0:
            raise ValueError("Jumlah transfer harus lebih dari 0")
        # Kedua lock selalu diambil berdasarkan urutan id objek.
        # Ini mencegah deadlock ketika dua rekening transfer dua arah.
        rekening_pertama, rekening_kedua = sorted(
            [self, rekening_tujuan], key=id
        )
        with rekening_pertama.lock:
            with rekening_kedua.lock:
                self.tarik(jumlah)                 # mengambil RLock asal lagi
                rekening_tujuan.setor(jumlah)     # mengambil RLock tujuan lagi
                print(f"{self.nama} -> {rekening_tujuan.nama}: Rp{jumlah:,}")

    def lihat_saldo(self):
        with self.lock:
            return self.saldo

rekening_a = RekeningBank("A", 500_000)
rekening_b = RekeningBank("B", 200_000)
print("Saldo awal:", rekening_a.lihat_saldo(), rekening_b.lihat_saldo())
rekening_a.transfer(rekening_b, 100_000)
rekening_b.transfer(rekening_a, 50_000)
print("Saldo akhir:", rekening_a.lihat_saldo(), rekening_b.lihat_saldo())
assert (rekening_a.lihat_saldo(), rekening_b.lihat_saldo()) == (450_000, 250_000)
assert rekening_a.lihat_saldo() + rekening_b.lihat_saldo() == 700_000


Saldo awal: 500000 200000
A -> B: Rp100,000
B -> A: Rp50,000
Saldo akhir: 450000 250000


RLock digunakan karena saat menjalankan transfer(), lock pada rekening sudah diambil, kemudian method tersebut memanggil tarik() atau setor() yang juga membutuhkan lock yang sama. Jika menggunakan Lock biasa, thread yang sedang memegang lock tidak dapat mengambilnya kembali sehingga dapat menyebabkan program berhenti menunggu (self-deadlock). Berbeda dengan itu, RLock memungkinkan thread yang sama mengambil lock secara berulang. Selain itu, penguncian kedua rekening dilakukan berdasarkan urutan yang sama untuk menghindari deadlock saat terjadi transfer dua arah. Apabila saldo tidak mencukupi, proses penarikan akan dibatalkan sehingga setoran ke rekening tujuan tidak dilakukan.

## 2. Deadlock dengan 3 Lock

Buat skenario deadlock yang melibatkan tiga lock dan tiga thread (bukan dua seperti contoh di notebook), dengan pola
circular wait: Thread1 menunggu Lock-B (dipegang Thread2), Thread2 menunggu Lock-C (dipegang Thread3), Thread3 menunggu Lock-A (dipegang
Thread1). Buktikan terjadinya deadlock menggunakan teknik subprocess + timeout seperti pada notebook ini, lalu perbaiki dengan urutan lock
yang konsisten.

In [2]:
import subprocess
import sys
import textwrap

kode_deadlock = textwrap.dedent(r"""
import threading

lock_a = threading.Lock()
lock_b = threading.Lock()
lock_c = threading.Lock()
barrier = threading.Barrier(3)

def tugas(nama, lock_pertama, lock_kedua, nama_kedua):
    with lock_pertama:
        print(f"{nama} memegang lock pertama", flush=True)
        barrier.wait()  # pastikan ketiga thread sudah memegang lock pertama
        print(f"{nama} menunggu {nama_kedua}", flush=True)
        with lock_kedua:
            print(f"{nama} selesai", flush=True)

threads = [
    threading.Thread(target=tugas, args=("Thread1", lock_a, lock_b, "Lock-B")),
    threading.Thread(target=tugas, args=("Thread2", lock_b, lock_c, "Lock-C")),
    threading.Thread(target=tugas, args=("Thread3", lock_c, lock_a, "Lock-A")),
]
for t in threads:
    t.start()
for t in threads:
    t.join()
""")

try:
    proses = subprocess.run(
        [sys.executable, "-u", "-c", kode_deadlock],
        capture_output=True, text=True, timeout=2
    )
    print(proses.stdout)
    print("Proses selesai (tidak terjadi deadlock)")
except subprocess.TimeoutExpired as error:
    keluaran = error.stdout or b""
    if isinstance(keluaran, bytes):
        keluaran = keluaran.decode("utf-8", errors="replace")
    print(keluaran)
    print("DEADLOCK TERBUKTI: ketiga thread tidak selesai dalam 2 detik.")
    print("Subprocess dihentikan otomatis, kernel notebook tetap berjalan.")


Thread1 memegang lock pertamaThread2 memegang lock pertama

Thread3 memegang lock pertama
Thread3 menunggu Lock-A
Thread2 menunggu Lock-C
Thread1 menunggu Lock-B

DEADLOCK TERBUKTI: ketiga thread tidak selesai dalam 2 detik.
Subprocess dihentikan otomatis, kernel notebook tetap berjalan.


**Perbaikan:** Setiap thread perlu menggunakan urutan pengambilan lock yang sama, yaitu A → B → C. Dengan menerapkan urutan tersebut, tidak ada thread yang memegang lock tertentu sambil menunggu lock sebelumnya yang sedang digunakan thread lain. Hal ini dapat memutus kondisi circular wait sehingga kemungkinan terjadinya deadlock dapat dihindari.

In [3]:
import threading
import time

lock_a = threading.Lock()
lock_b = threading.Lock()
lock_c = threading.Lock()

# Semua thread mengambil lock sesuai urutan global A, B, C.
urutan_lock = {"A": (0, lock_a), "B": (1, lock_b), "C": (2, lock_c)}

def tugas_aman(nama, lock_dibutuhkan):
    lock_urut = sorted((urutan_lock[x] for x in lock_dibutuhkan), key=lambda x: x[0])
    for _, lock in lock_urut:
        lock.acquire()
    try:
        print(f"{nama} berhasil memegang {', '.join(sorted(lock_dibutuhkan))}")
        time.sleep(0.05)
    finally:
        for _, lock in reversed(lock_urut):
            lock.release()
    print(f"{nama} selesai")

threads = [
    threading.Thread(target=tugas_aman, args=("Thread1", ["A", "B"])),
    threading.Thread(target=tugas_aman, args=("Thread2", ["B", "C"])),
    threading.Thread(target=tugas_aman, args=("Thread3", ["C", "A"])),
]
for t in threads:
    t.start()
for t in threads:
    t.join(timeout=3)
assert all(not t.is_alive() for t in threads), "Ada thread yang belum selesai"
print("Tidak ada deadlock: semua thread selesai.")


Thread1 berhasil memegang A, B
Thread1 selesai
Thread2 berhasil memegang B, C
Thread2 selesai
Thread3 berhasil memegang A, C
Thread3 selesai
Tidak ada deadlock: semua thread selesai.


**Kesimpulan:** Pada kondisi awal, program mengalami kemacetan karena masing-masing thread saling menunggu lock yang masih digunakan oleh thread lainnya. Setelah urutan pengambilan lock diseragamkan, seluruh thread dapat menyelesaikan prosesnya. Penggunaan timeout pada subprocess hanya berfungsi untuk mendeteksi adanya deadlock melalui batas waktu eksekusi, bukan sebagai cara untuk mengatasi penyebab deadlock tersebut.

## 3. Bounded Buffer dengan Kapasitas Berbeda

Ulangi eksperimen bagian 7 dengan UKURAN_BUFFER = 1, 3, 10, dan 2 producer + 2 consumer
sekaligus (bukan 1 producer). Amati dan jelaskan bagaimana ukuran buffer mempengaruhi pola interleaving antara producer dan consumer pada
output.

In [4]:
import threading
import time


def jalankan_buffer(ukuran_buffer, item_per_producer=10):
    buffer = []
    mutex = threading.Lock()
    slot_kosong = threading.Semaphore(ukuran_buffer)
    slot_terisi = threading.Semaphore(0)

    hasil_produksi = []
    hasil_konsumsi = []
    waktu_tunggu_producer = []
    waktu_tunggu_consumer = []
    kapasitas_tertinggi = [0]
    catatan = []

    def producer(nama):
        for i in range(item_per_producer):
            item = f"{nama}-item-{i}"
            time.sleep(0.002)  # produksi relatif cepat
            awal = time.perf_counter()
            slot_kosong.acquire()  # menunggu jika buffer penuh
            tunggu = time.perf_counter() - awal
            with mutex:
                buffer.append(item)
                hasil_produksi.append(item)
                kapasitas_tertinggi[0] = max(kapasitas_tertinggi[0], len(buffer))
                if len(catatan) < 8:
                    catatan.append(f"[{nama}] taruh {item} (buffer: {len(buffer)}/{ukuran_buffer})")
                waktu_tunggu_producer.append(tunggu)
            slot_terisi.release()

    def consumer(nama):
        for _ in range(item_per_producer):
            awal = time.perf_counter()
            slot_terisi.acquire()  # menunggu jika buffer kosong
            tunggu = time.perf_counter() - awal
            with mutex:
                item = buffer.pop(0)
                hasil_konsumsi.append(item)
                if len(catatan) < 8:
                    catatan.append(f"[{nama}] ambil {item} (buffer: {len(buffer)}/{ukuran_buffer})")
                waktu_tunggu_consumer.append(tunggu)
            slot_kosong.release()
            time.sleep(0.018)  # konsumsi lebih lambat daripada produksi

    threads = [
        threading.Thread(target=producer, args=("Producer-1",)),
        threading.Thread(target=producer, args=("Producer-2",)),
        threading.Thread(target=consumer, args=("Consumer-1",)),
        threading.Thread(target=consumer, args=("Consumer-2",)),
    ]
    awal_percobaan = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join(timeout=5)
    durasi = time.perf_counter() - awal_percobaan

    assert all(not t.is_alive() for t in threads), "Ada thread belum selesai"
    assert len(buffer) == 0, "Buffer seharusnya kosong"
    assert len(hasil_produksi) == len(hasil_konsumsi) == 2 * item_per_producer
    assert sorted(hasil_produksi) == sorted(hasil_konsumsi)
    assert kapasitas_tertinggi[0] <= ukuran_buffer

    return {
        "kapasitas": ukuran_buffer,
        "maks_terisi": kapasitas_tertinggi[0],
        "total_item": len(hasil_konsumsi),
        "durasi": durasi,
        "tunggu_producer": sum(waktu_tunggu_producer),
        "tunggu_consumer": sum(waktu_tunggu_consumer),
        "producer_tertahan": sum(w > 0.003 for w in waktu_tunggu_producer),
        "log": catatan,
    }

hasil_percobaan = [jalankan_buffer(k) for k in (1, 3, 10)]
for hasil in hasil_percobaan:
    print(f"\n=== UKURAN_BUFFER = {hasil['kapasitas']} ===")
    for baris in hasil["log"]:
        print(baris)
    print(f"Total dikonsumsi: {hasil['total_item']} | Maks. isi: {hasil['maks_terisi']}")
    print(f"Total waktu tunggu producer: {hasil['tunggu_producer']:.4f} detik")
    print(f"Producer tertahan > 3 ms: {hasil['producer_tertahan']} kali")
    print(f"Sisa buffer: 0 | Durasi: {hasil['durasi']:.4f} detik")



=== UKURAN_BUFFER = 1 ===
[Producer-1] taruh Producer-1-item-0 (buffer: 1/1)
[Consumer-1] ambil Producer-1-item-0 (buffer: 0/1)
[Producer-2] taruh Producer-2-item-0 (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-0 (buffer: 0/1)
[Producer-1] taruh Producer-1-item-1 (buffer: 1/1)
[Consumer-1] ambil Producer-1-item-1 (buffer: 0/1)
[Producer-2] taruh Producer-2-item-1 (buffer: 1/1)
[Consumer-2] ambil Producer-2-item-1 (buffer: 0/1)
Total dikonsumsi: 20 | Maks. isi: 1
Total waktu tunggu producer: 0.2712 detik
Producer tertahan > 3 ms: 17 kali
Sisa buffer: 0 | Durasi: 0.1849 detik

=== UKURAN_BUFFER = 3 ===
[Producer-1] taruh Producer-1-item-0 (buffer: 1/3)
[Consumer-1] ambil Producer-1-item-0 (buffer: 0/3)
[Producer-2] taruh Producer-2-item-0 (buffer: 1/3)
[Consumer-2] ambil Producer-2-item-0 (buffer: 0/3)
[Producer-1] taruh Producer-1-item-1 (buffer: 1/3)
[Producer-2] taruh Producer-2-item-1 (buffer: 2/3)
[Producer-1] taruh Producer-1-item-2 (buffer: 3/3)
[Consumer-1] ambil Producer-1-i

### Ringkasan perbandingan

Lama waktu tunggu diperoleh dari durasi producer saat menunggu slot_kosong.acquire() dan consumer ketika menunggu slot_terisi.acquire(). Hasil pengukuran tersebut dapat sedikit berbeda setiap kali program dijalankan karena proses penjadwalan thread tidak selalu berlangsung dalam urutan yang sama.

In [5]:
print(f"{'Buffer':>8} | {'Maks isi':>8} | {'Tunggu producer (s)':>20} | {'Tertahan >3ms':>13} | {'Item selesai':>12}")
print("-" * 78)
for h in hasil_percobaan:
    print(f"{h['kapasitas']:>8} | {h['maks_terisi']:>8} | {h['tunggu_producer']:>20.4f} | {h['producer_tertahan']:>13} | {h['total_item']:>12}")


  Buffer | Maks isi |  Tunggu producer (s) | Tertahan >3ms | Item selesai
------------------------------------------------------------------------------
       1 |        1 |               0.2712 |            17 |           20
       3 |        3 |               0.2344 |            15 |           20
      10 |       10 |               0.1079 |             8 |           20


- Buffer 1: Producer lebih sering mengalami waktu tunggu karena kapasitas buffer hanya satu item. Ketika slot tersebut sudah terisi, producer harus menunggu hingga consumer mengambil item terlebih dahulu.
- Buffer 3: Producer akan menunggu apabila ketiga slot telah terisi. Namun, kapasitas ini menyediakan ruang penyimpanan sementara yang lebih banyak dibandingkan buffer 1.
- Buffer 10: Producer memiliki ruang lebih luas untuk memasukkan item sebelum harus menunggu. Dengan demikian, waktu tunggu producer cenderung lebih singkat, meskipun waktu penyelesaian seluruh proses tetap dipengaruhi oleh kecepatan consumer.

Pada ketiga variasi kapasitas tersebut, seluruh 20 item berhasil diproses oleh consumer dan tidak ada item yang tersisa di dalam buffer. Jumlah item yang masuk juga tetap berada dalam batas kapasitas yang telah ditentukan. Perbedaan hasil pengukuran pada masing-masing kapasitas dapat dilihat pada tabel output.

## 4. Refleksi Singkat

Dari ketiga masalah (deadlock, livelock, starvation), menurut kalian mana yang paling berbahaya untuk aplikasi
produksi nyata (misalnya sistem perbankan atau e-commerce), dan mana yang paling sulit dideteksi saat testing? Jelaskan alasannya.

Menurut saya, deadlock menjadi salah satu masalah yang paling berisiko dalam aplikasi produksi, seperti layanan perbankan dan e-commerce. Kondisi ini dapat menyebabkan beberapa proses terhenti karena masing-masing saling menunggu resource yang sedang digunakan. Sebagai contoh, proses transfer dana atau pembaruan stok barang dapat tertunda sehingga pengguna tidak dapat menyelesaikan transaksi. Oleh karena itu, deadlock perlu segera diketahui karena terkadang aplikasi hanya terlihat berjalan lambat, padahal proses di dalamnya sudah tidak mengalami kemajuan. Untuk mencegahnya, pengambilan lock perlu diatur dengan urutan yang konsisten, disertai pembatasan waktu tunggu dan pemantauan terhadap proses yang berlangsung terlalu lama. Meskipun livelock dan starvation juga dapat menghambat kinerja aplikasi, deadlock memiliki dampak yang lebih langsung karena dapat menghentikan proses secara keseluruhan.